In [38]:
import sys
!{sys.executable} -m pip install rdflib owlrl

# utilities
import pandas as pd

# libraries to handle triples and graphs
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDFS, RDF, OWL

# libraries to handle reasoning
from owlrl import DeductiveClosure, OWLRL_Semantics


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [39]:
# utility variable : list of namespaces that we will need for querying
namespaces = {'rdf': RDF, 'rdfs' : RDFS, '': 'http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/', 'owl': OWL}
hi = Namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")



In [40]:
# initialise an empty graph
g = Graph()

# parse a knowledge graph from a local file
g = Graph().parse("hi_ontology_updated_amee_max.owl")

In [41]:
g.bind("hi", hi)

In [42]:
print("Graph has %s statements." % len(g))

Graph has 551 statements.


In [ ]:
# Define inverse relationship
g.add((hi.hasActor, RDF.type, OWL.ObjectProperty))
g.add((hi.hasActor, OWL.inverseOf, hi.inScenario))

<Graph identifier=N7f8d532545b047b6a0b6d2c785e318c0 (<class 'rdflib.graph.Graph'>)>

In [31]:
for s in g.objects(None,hi.hasActor):
  print(s)


In [ ]:
# initialise new graphs : asserted (triples stated in the ttl file), inferred (triples generated by the reasoner)
asserted = Graph()
inferred = Graph()

asserted = asserted.parse("onto_with_inst/hi_ontology_amneepapers.owl")
DeductiveClosure(OWLRL_Semantics).expand(g) 
inferred = g - asserted

In [33]:
print("asserted {}, inferred {}, total {}".format(len(asserted), len(inferred),len(g)))


asserted 324, inferred 492, total 816


In [ ]:
from owlready2 import *

# Load your ontology
onto = get_ontology("file://hi_ontology_updated_amee_max.owl").load()

# Reuse the ontology namespace
hi = onto.get_namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")

with onto:

    # ---------- New classes ----------
    class TrustLevel(Thing):
        namespace = hi

    class HighTrust(TrustLevel):
        namespace = hi

    class MediumTrust(TrustLevel):
        namespace = hi

    class LowTrust(TrustLevel):
        namespace = hi

    class AutonomyLevel(Thing):
        namespace = hi

    class LowAutonomy(AutonomyLevel):
        namespace = hi

    class SemiAutonomous(AutonomyLevel):
        namespace = hi

    class HighAutonomy(AutonomyLevel):
        namespace = hi

    class ControlMode(Thing):
        namespace = hi

    class HumanInTheLoop(ControlMode):
        namespace = hi

    class HumanOnTheLoop(ControlMode):
        namespace = hi

    class HumanOutOfTheLoop(ControlMode):
        namespace = hi

    class DecisionRole(Thing):
        namespace = hi

    class HumanFinalDecisionMaker(DecisionRole):
        namespace = hi

    class AIFinalDecisionMaker(DecisionRole):
        namespace = hi

    class SharedDecisionMaking(DecisionRole):
        namespace = hi

    class ExplanationType(Thing):
        namespace = hi

    class OutcomeExplanation(ExplanationType):
        namespace = hi

    class ProcessExplanation(ExplanationType):
        namespace = hi

    class CounterfactualExplanation(ExplanationType):
        namespace = hi

    class CoordinationPattern(Thing):
        namespace = hi

    class SequentialCoordination(CoordinationPattern):
        namespace = hi

    class ParallelCoordination(CoordinationPattern):
        namespace = hi

    class SupervisoryCoordination(CoordinationPattern):
        namespace = hi
    
    class CollaborativeCoordination(CoordinationPattern):
        namespace = hi

    # ---------- New object properties ----------
    class hasTrustLevel(ObjectProperty):
        namespace = hi
        domain = [hi.Interaction]
        range = [TrustLevel]

    class trustLevelOf(ObjectProperty):
        namespace = hi
        domain = [TrustLevel]
        range = [hi.Interaction]
        inverse_property = hasTrustLevel

    class hasAutonomyLevel(ObjectProperty):
        namespace = hi
        domain = [hi.ArtificialAgent]
        range = [AutonomyLevel]

    class autonomyLevelOf(ObjectProperty):
        namespace = hi
        domain = [AutonomyLevel]
        range = [hi.ArtificialAgent]
        inverse_property = hasAutonomyLevel
    
    class hasActor(ObjectProperty):
        namespace = hi
        domain = [hi.Scenario]
        range = [hi.Actor]

    hasActor.inverse_property = hi.inScenario

    class hasControlMode(ObjectProperty):
        namespace = hi
        domain = [hi.Interaction]
        range = [ControlMode]

    class controlModeOf(ObjectProperty):
        namespace = hi
        domain = [ControlMode]
        range = [hi.Interaction]
        inverse_property = hasControlMode

    class hasDecisionRole(ObjectProperty):
        namespace = hi
        domain = [hi.Actor]
        range = [DecisionRole]

    class decisionRoleOf(ObjectProperty):
        namespace = hi
        domain = [DecisionRole]
        range = [hi.Actor]
        inverse_property = hasDecisionRole

    class providesExplanationType(ObjectProperty):
        namespace = hi
        domain = [hi.ArtificialAgent]
        range = [ExplanationType]

    class hasCoordinationPattern(ObjectProperty):
        namespace = hi
        domain = [hi.Interaction]
        range = [CoordinationPattern]

    class collaboratesWith(ObjectProperty, SymmetricProperty):
        namespace = hi
        domain = [hi.Actor]
        range = [hi.Actor]

    class broaderCapability(ObjectProperty, TransitiveProperty):
        namespace = hi
        domain = [hi.Capability]
        range = [hi.Capability]
    
    class subjectToControlMode(ObjectProperty):
        namespace = hi
        domain = [hi.ArtificialAgent] # this property only consider for artificial agents
        range = [ControlMode]

    # ---------- Disjointness ----------
    AllDisjoint([hi.Human, hi.ArtificialAgent])
    AllDisjoint([HighTrust, MediumTrust, LowTrust])
    AllDisjoint([LowAutonomy, SemiAutonomous, HighAutonomy])
    AllDisjoint([HumanInTheLoop, HumanOnTheLoop, HumanOutOfTheLoop])

    # ---------- Union class ----------
    class DecisionParticipant(Thing):
        namespace = hi
        equivalent_to = [hi.Human | hi.ArtificialAgent]

    # ---------- Equivalent classes / intersections ----------
    class HighTrustInteraction(hi.Interaction):
        namespace = hi
        equivalent_to = [hi.Interaction & hasTrustLevel.some(HighTrust)]
    

    class SupervisedAIAgent(hi.ArtificialAgent):
        namespace = hi
        equivalent_to = [hi.ArtificialAgent & subjectToControlMode.some(HumanOnTheLoop)]

    # ---------- Restrictions ----------
    hi.ArtificialAgent.is_a.append(hasAutonomyLevel.exactly(1, AutonomyLevel))
    hi.Interaction.is_a.append(hasTrustLevel.only(TrustLevel))
    hi.Interaction.is_a.append(hasCoordinationPattern.only(CoordinationPattern))
    hi.Actor.is_a.append(hasDecisionRole.only(DecisionRole))

    # If your ontology already has interactingAgent, uncomment this:
    hi.Interaction.is_a.append(hi.interactingAgent.min(2, hi.Actor))

# Save
onto.save(file="hi_ontology_extended.owl", format="rdfxml")
onto.save(file="hi_ontology_extended.ttl", format="ntriples")

In [ ]:
from owlready2 import *

# Load your ontology
onto = get_ontology("file://hi_ontology_updated_amee_max.owl").load()

# Reuse the ontology namespace
hi = onto.get_namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")



# ---------- New classes ----------
class TrustLevel(Thing):
    namespace = hi

class HighTrust(TrustLevel):
    namespace = hi

class MediumTrust(TrustLevel):
    namespace = hi

class LowTrust(TrustLevel):
    namespace = hi

class AutonomyLevel(Thing):
    namespace = hi

class LowAutonomy(AutonomyLevel):
    namespace = hi

class SemiAutonomous(AutonomyLevel):
    namespace = hi

class HighAutonomy(AutonomyLevel):
    namespace = hi

class ControlMode(Thing):
    namespace = hi

class HumanInTheLoop(ControlMode):
    namespace = hi

class HumanOnTheLoop(ControlMode):
    namespace = hi

class HumanOutOfTheLoop(ControlMode):
    namespace = hi

class DecisionRole(Thing):
    namespace = hi

class HumanFinalDecisionMaker(DecisionRole):
    namespace = hi

class AIFinalDecisionMaker(DecisionRole):
    namespace = hi

class SharedDecisionMaking(DecisionRole):
    namespace = hi

class ExplanationType(Thing):
    namespace = hi

class OutcomeExplanation(ExplanationType):
    namespace = hi

class ProcessExplanation(ExplanationType):
    namespace = hi

class CounterfactualExplanation(ExplanationType):
    namespace = hi

class CoordinationPattern(Thing):
    namespace = hi

class SequentialCoordination(CoordinationPattern):
    namespace = hi

class ParallelCoordination(CoordinationPattern):
    namespace = hi

class SupervisoryCoordination(CoordinationPattern):
    namespace = hi

class CollaborativeCoordination(CoordinationPattern):
    namespace = hi

# ---------- New object properties ----------
class hasTrustLevel(ObjectProperty):
    namespace = hi
    domain = [hi.Interaction]
    range = [TrustLevel]

class trustLevelOf(ObjectProperty):
    namespace = hi
    domain = [TrustLevel]
    range = [hi.Interaction]
    inverse_property = hasTrustLevel

class hasAutonomyLevel(ObjectProperty):
    namespace = hi
    domain = [hi.ArtificialAgent]
    range = [AutonomyLevel]

class autonomyLevelOf(ObjectProperty):
    namespace = hi
    domain = [AutonomyLevel]
    range = [hi.ArtificialAgent]
    inverse_property = hasAutonomyLevel

class hasActor(ObjectProperty):
    namespace = hi
    domain = [hi.Scenario]
    range = [hi.Actor]

hasActor.inverse_property = hi.inScenario

class hasControlMode(ObjectProperty):
    namespace = hi
    domain = [hi.Interaction]
    range = [ControlMode]

class controlModeOf(ObjectProperty):
    namespace = hi
    domain = [ControlMode]
    range = [hi.Interaction]
    inverse_property = hasControlMode

class hasDecisionRole(ObjectProperty):
    namespace = hi
    domain = [hi.Actor]
    range = [DecisionRole]

class decisionRoleOf(ObjectProperty):
    namespace = hi
    domain = [DecisionRole]
    range = [hi.Actor]
    inverse_property = hasDecisionRole

class providesExplanationType(ObjectProperty):
    namespace = hi
    domain = [hi.ArtificialAgent]
    range = [ExplanationType]

class hasCoordinationPattern(ObjectProperty):
    namespace = hi
    domain = [hi.Interaction]
    range = [CoordinationPattern]

class collaboratesWith(ObjectProperty, SymmetricProperty):
    namespace = hi
    domain = [hi.Actor]
    range = [hi.Actor]

class broaderCapability(ObjectProperty, TransitiveProperty):
    namespace = hi
    domain = [hi.Capability]
    range = [hi.Capability]

class subjectToControlMode(ObjectProperty):
    namespace = hi
    domain = [hi.ArtificialAgent] # this property only consider for artificial agents
    range = [ControlMode]

# ---------- Disjointness ----------
AllDisjoint([hi.Human, hi.ArtificialAgent])
AllDisjoint([HighTrust, MediumTrust, LowTrust])
AllDisjoint([LowAutonomy, SemiAutonomous, HighAutonomy])
AllDisjoint([HumanInTheLoop, HumanOnTheLoop, HumanOutOfTheLoop])

# ---------- Union class ----------
class DecisionParticipant(Thing):
    namespace = hi
    equivalent_to = [hi.Human | hi.ArtificialAgent]

# ---------- Equivalent classes / intersections ----------
class HighTrustInteraction(hi.Interaction):
    namespace = hi
    equivalent_to = [hi.Interaction & hasTrustLevel.some(HighTrust)]


class SupervisedAIAgent(hi.ArtificialAgent):
    namespace = hi
    equivalent_to = [hi.ArtificialAgent & subjectToControlMode.some(HumanOnTheLoop)]

# ---------- Restrictions ----------
hi.ArtificialAgent.is_a.append(hasAutonomyLevel.exactly(1, AutonomyLevel))
hi.Interaction.is_a.append(hasTrustLevel.only(TrustLevel))
hi.Interaction.is_a.append(hasCoordinationPattern.only(CoordinationPattern))
hi.Actor.is_a.append(hasDecisionRole.only(DecisionRole))

# If your ontology already has interactingAgent, uncomment this:
hi.Interaction.is_a.append(hi.interactingAgent.min(2, hi.Actor))

# Save
onto.save(file="hi_ontology_extended.owl", format="rdfxml")
onto.save(file="hi_ontology_extended.ttl", format="ntriples")

### Add instances

In [80]:
from owlready2 import *

# Load ontology
onto = get_ontology("file://hi_ontology_extended.owl").load()
hi = onto.get_namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")

with onto:
    # ==================================================
    # SCENARIOS
    # ==================================================
    clinical_scenario_1 = hi.Scenario("clinical_scenario_1")
    legal_scenario_1 = hi.Scenario("legal_scenario_1")
    warehouse_scenario_1 = hi.Scenario("warehouse_scenario_1")

    # ==================================================
    # HUMANS
    # ==================================================
    doctor_1 = hi.Human("doctor_1")
    nurse_1 = hi.Human("nurse_1")
    patient_1 = hi.Human("patient_1")
    legal_advisor_1 = hi.Human("legal_advisor_1")
    warehouse_worker_1 = hi.Human("warehouse_worker_1")
    supervisor_1 = hi.Human("supervisor_1")

    # ==================================================
    # ARTIFICIAL AGENTS
    # ==================================================
    diagnosis_ai_1 = hi.ArtificialAgent("diagnosis_ai_1")
    treatment_ai_1 = hi.ArtificialAgent("treatment_ai_1")
    legal_ai_1 = hi.ArtificialAgent("legal_ai_1")
    warehouse_agv_1 = hi.ArtificialAgent("warehouse_agv_1")
    monitoring_ai_1 = hi.ArtificialAgent("monitoring_ai_1")

    # ==================================================
    # INTERACTIONS
    # ==================================================
    interaction_1 = hi.Interaction("interaction_1")
    interaction_2 = hi.Interaction("interaction_2")
    interaction_3 = hi.Interaction("interaction_3")
    interaction_4 = hi.Interaction("interaction_4")
    interaction_5 = hi.Interaction("interaction_5")
    interaction_6 = hi.Interaction("interaction_6")

    # ==================================================
    # TRUST LEVELS
    # ==================================================
    high_trust_case_1 = hi.HighTrust("high_trust_case_1")
    high_trust_case_2 = hi.HighTrust("high_trust_case_2")
    medium_trust_case_1 = hi.MediumTrust("medium_trust_case_1")
    medium_trust_case_2 = hi.MediumTrust("medium_trust_case_2")
    low_trust_case_1 = hi.LowTrust("low_trust_case_1")

    # ==================================================
    # AUTONOMY LEVELS
    # ==================================================
    low_auto_case_1 = hi.LowAutonomy("low_auto_case_1")
    semi_auto_case_1 = hi.SemiAutonomous("semi_auto_case_1")
    semi_auto_case_2 = hi.SemiAutonomous("semi_auto_case_2")
    high_auto_case_1 = hi.HighAutonomy("high_auto_case_1")

    # ==================================================
    # CONTROL MODES
    # ==================================================
    hitl_mode_1 = hi.HumanInTheLoop("hitl_mode_1")
    hitl_mode_2 = hi.HumanInTheLoop("hitl_mode_2")
    hotl_mode_1 = hi.HumanOnTheLoop("hotl_mode_1")
    hootl_mode_1 = hi.HumanOutOfTheLoop("hootl_mode_1")

    # ==================================================
    # EXPLANATION TYPES
    # ==================================================
    outcome_expl_1 = hi.OutcomeExplanation("outcome_expl_1")
    outcome_expl_2 = hi.OutcomeExplanation("outcome_expl_2")
    process_expl_1 = hi.ProcessExplanation("process_expl_1")
    process_expl_2 = hi.ProcessExplanation("process_expl_2")

    # ==================================================
    # DECISION ROLES
    # ==================================================
    human_final_role_1 = hi.HumanFinalDecisionMaker("human_final_role_1")
    human_final_role_2 = hi.HumanFinalDecisionMaker("human_final_role_2")
    ai_support_role_1 = hi.AIFinalDecisionMaker("ai_support_role_1")
    ai_support_role_2 = hi.AIFinalDecisionMaker("ai_support_role_2")



    # ==================================================
    # COORDINATION PATTERNS
    # ==================================================
    seq_coord_1 = hi.SequentialCoordination("seq_coord_1")
    seq_coord_2 = hi.SequentialCoordination("seq_coord_2")
    collab_coord_1 = hi.CollaborativeCoordination("collab_coord_1")
    collab_coord_2 = hi.CollaborativeCoordination("collab_coord_2")

    # ==================================================
    # CAPABILITIES
    # ==================================================
    prediction_cap_1 = hi.Capability("prediction_cap_1")
    recommendation_cap_1 = hi.Capability("recommendation_cap_1")
    explanation_cap_1 = hi.Capability("explanation_cap_1")
    monitoring_cap_1 = hi.Capability("monitoring_cap_1")
    navigation_cap_1 = hi.Capability("navigation_cap_1")
    legal_reasoning_cap_1 = hi.Capability("legal_reasoning_cap_1")

    # ==================================================
    # INTERACTION METHODS
    # ==================================================
    dashboard_method_1 = hi.InteractionMethod("dashboard_method_1")
    alert_method_1 = hi.InteractionMethod("alert_method_1")
    chat_method_1 = hi.InteractionMethod("chat_method_1")
    voice_method_1 = hi.InteractionMethod("voice_method_1")

    # ==================================================
    # INTERACTION TASKS
    # ==================================================
    diagnosis_task_1 = hi.InteractionTask("diagnosis_task_1")
    treatment_task_1 = hi.InteractionTask("treatment_task_1")
    legal_review_task_1 = hi.InteractionTask("legal_review_task_1")
    warehouse_navigation_task_1 = hi.InteractionTask("warehouse_navigation_task_1")
    monitoring_task_1 = hi.InteractionTask("monitoring_task_1")

    # ==================================================
    # CONTEXTS
    # ==================================================
    hospital_context_1 = hi.Context("hospital_context_1")
    emergency_context_1 = hi.Context("emergency_context_1")
    legal_office_context_1 = hi.Context("legal_office_context_1")
    warehouse_context_1 = hi.Context("warehouse_context_1")

    # ==================================================
    # ENDGOALS
    # ==================================================
    patient_safety_goal_1 = hi.Endgoal("patient_safety_goal_1")
    diagnosis_accuracy_goal_1 = hi.Endgoal("diagnosis_accuracy_goal_1")
    legal_efficiency_goal_1 = hi.Endgoal("legal_efficiency_goal_1")
    warehouse_safety_goal_1 = hi.Endgoal("warehouse_safety_goal_1")

    # ==================================================
    # DOMAINS
    # ==================================================
    healthcare_domain_1 = hi.Domain("healthcare_domain_1")
    legal_domain_1 = hi.Domain("legal_domain_1")
    logistics_domain_1 = hi.Domain("logistics_domain_1")

    # ==================================================
    # ETHICAL CONSIDERATIONS
    # ==================================================
    transparency_ethics_1 = hi.EthicalConsideration("transparency_ethics_1")
    safety_ethics_1 = hi.EthicalConsideration("safety_ethics_1")
    fairness_ethics_1 = hi.EthicalConsideration("fairness_ethics_1")
    accountability_ethics_1 = hi.EthicalConsideration("accountability_ethics_1")

    # ==================================================
    # ACTOR-LEVEL RELATIONS
    # ==================================================
    diagnosis_ai_1.hasAutonomyLevel = [semi_auto_case_1]
    treatment_ai_1.hasAutonomyLevel = [semi_auto_case_2]
    legal_ai_1.hasAutonomyLevel = [low_auto_case_1]
    warehouse_agv_1.hasAutonomyLevel = [high_auto_case_1]
    monitoring_ai_1.hasAutonomyLevel = [semi_auto_case_1]

    diagnosis_ai_1.providesExplanationType = [outcome_expl_1, process_expl_1]
    treatment_ai_1.providesExplanationType = [outcome_expl_2]
    legal_ai_1.providesExplanationType = [process_expl_2]
    monitoring_ai_1.providesExplanationType = [outcome_expl_1]

    doctor_1.hasDecisionRole = [human_final_role_1]
    legal_advisor_1.hasDecisionRole = [human_final_role_2]
    diagnosis_ai_1.hasDecisionRole = [ai_support_role_1]
    legal_ai_1.hasDecisionRole = [ai_support_role_2]
    treatment_ai_1.hasDecisionRole = [ai_support_role_1]

    doctor_1.collaboratesWith.append(nurse_1)
    doctor_1.collaboratesWith.append(diagnosis_ai_1)
    doctor_1.collaboratesWith.append(treatment_ai_1)
    nurse_1.collaboratesWith.append(diagnosis_ai_1)
    nurse_1.collaboratesWith.append(monitoring_ai_1)
    legal_advisor_1.collaboratesWith.append(legal_ai_1)
    warehouse_worker_1.collaboratesWith.append(warehouse_agv_1)
    supervisor_1.collaboratesWith.append(warehouse_agv_1)
    supervisor_1.collaboratesWith.append(monitoring_ai_1)

    doctor_1.capability.append(prediction_cap_1)
    doctor_1.capability.append(recommendation_cap_1)
    nurse_1.capability.append(monitoring_cap_1)
    legal_advisor_1.capability.append(legal_reasoning_cap_1)
    warehouse_worker_1.capability.append(navigation_cap_1)
    supervisor_1.capability.append(monitoring_cap_1)

    diagnosis_ai_1.capability.append(prediction_cap_1)
    diagnosis_ai_1.capability.append(explanation_cap_1)
    treatment_ai_1.capability.append(recommendation_cap_1)
    treatment_ai_1.capability.append(explanation_cap_1)
    legal_ai_1.capability.append(legal_reasoning_cap_1)
    legal_ai_1.capability.append(explanation_cap_1)
    warehouse_agv_1.capability.append(navigation_cap_1)
    warehouse_agv_1.capability.append(monitoring_cap_1)
    monitoring_ai_1.capability.append(monitoring_cap_1)
    monitoring_ai_1.capability.append(explanation_cap_1)

    doctor_1.inScenario.append(clinical_scenario_1)
    nurse_1.inScenario.append(clinical_scenario_1)
    patient_1.inScenario.append(clinical_scenario_1)
    diagnosis_ai_1.inScenario.append(clinical_scenario_1)
    treatment_ai_1.inScenario.append(clinical_scenario_1)
    monitoring_ai_1.inScenario.append(clinical_scenario_1)

    legal_advisor_1.inScenario.append(legal_scenario_1)
    legal_ai_1.inScenario.append(legal_scenario_1)

    warehouse_worker_1.inScenario.append(warehouse_scenario_1)
    supervisor_1.inScenario.append(warehouse_scenario_1)
    warehouse_agv_1.inScenario.append(warehouse_scenario_1)
    monitoring_ai_1.inScenario.append(warehouse_scenario_1)

    # ==================================================
    # INTERACTION-LEVEL RELATIONS
    # ==================================================
    interaction_1.hasTrustLevel = [high_trust_case_1]
    interaction_1.hasCoordinationPattern = [seq_coord_1]
    interaction_1.hasControlMode = [hitl_mode_1]
    interaction_1.interactingAgent.append(doctor_1)
    interaction_1.interactingAgent.append(diagnosis_ai_1)
    interaction_1.interactionMethod.append(dashboard_method_1)
    interaction_1.interactionTask.append(diagnosis_task_1)

    interaction_2.hasTrustLevel = [medium_trust_case_1]
    interaction_2.hasCoordinationPattern = [collab_coord_1]
    interaction_2.hasControlMode = [hotl_mode_1]
    interaction_2.interactingAgent.append(nurse_1)
    interaction_2.interactingAgent.append(monitoring_ai_1)
    interaction_2.interactionMethod.append(alert_method_1)
    interaction_2.interactionTask.append(monitoring_task_1)

    interaction_3.hasTrustLevel = [high_trust_case_2]
    interaction_3.hasCoordinationPattern = [collab_coord_2]
    interaction_3.hasControlMode = [hitl_mode_2]
    interaction_3.interactingAgent.append(doctor_1)
    interaction_3.interactingAgent.append(treatment_ai_1)
    interaction_3.interactionMethod.append(chat_method_1)
    interaction_3.interactionTask.append(treatment_task_1)

    interaction_4.hasTrustLevel = [medium_trust_case_2]
    interaction_4.hasCoordinationPattern = [seq_coord_2]
    interaction_4.hasControlMode = [hitl_mode_2]
    interaction_4.interactingAgent.append(legal_advisor_1)
    interaction_4.interactingAgent.append(legal_ai_1)
    interaction_4.interactionMethod.append(chat_method_1)
    interaction_4.interactionTask.append(legal_review_task_1)

    interaction_5.hasTrustLevel = [low_trust_case_1]
    interaction_5.hasCoordinationPattern = [seq_coord_1]
    interaction_5.hasControlMode = [hotl_mode_1]
    interaction_5.interactingAgent.append(warehouse_worker_1)
    interaction_5.interactingAgent.append(warehouse_agv_1)
    interaction_5.interactionMethod.append(voice_method_1)
    interaction_5.interactionTask.append(warehouse_navigation_task_1)

    interaction_6.hasTrustLevel = [medium_trust_case_1]
    interaction_6.hasCoordinationPattern = [collab_coord_1]
    interaction_6.hasControlMode = [hootl_mode_1]
    interaction_6.interactingAgent.append(supervisor_1)
    interaction_6.interactingAgent.append(monitoring_ai_1)
    interaction_6.interactionMethod.append(alert_method_1)
    interaction_6.interactionTask.append(monitoring_task_1)

    # ==================================================
    # SCENARIO-LEVEL RELATIONS
    # ==================================================
    clinical_scenario_1.hasInteraction.append(interaction_1)
    clinical_scenario_1.hasInteraction.append(interaction_2)
    clinical_scenario_1.hasInteraction.append(interaction_3)
    clinical_scenario_1.context.append(hospital_context_1)
    clinical_scenario_1.context.append(emergency_context_1)
    clinical_scenario_1.endgoal.append(patient_safety_goal_1)
    clinical_scenario_1.endgoal.append(diagnosis_accuracy_goal_1)
    clinical_scenario_1.domain.append(healthcare_domain_1)
    clinical_scenario_1.hasEthicalConsideration.append(transparency_ethics_1)
    clinical_scenario_1.hasEthicalConsideration.append(safety_ethics_1)
    clinical_scenario_1.hasEthicalConsideration.append(accountability_ethics_1)

    legal_scenario_1.hasInteraction.append(interaction_4)
    legal_scenario_1.context.append(legal_office_context_1)
    legal_scenario_1.endgoal.append(legal_efficiency_goal_1)
    legal_scenario_1.domain.append(legal_domain_1)
    legal_scenario_1.hasEthicalConsideration.append(transparency_ethics_1)
    legal_scenario_1.hasEthicalConsideration.append(fairness_ethics_1)
    legal_scenario_1.hasEthicalConsideration.append(accountability_ethics_1)

    warehouse_scenario_1.hasInteraction.append(interaction_5)
    warehouse_scenario_1.hasInteraction.append(interaction_6)
    warehouse_scenario_1.context.append(warehouse_context_1)
    warehouse_scenario_1.endgoal.append(warehouse_safety_goal_1)
    warehouse_scenario_1.domain.append(logistics_domain_1)
    warehouse_scenario_1.hasEthicalConsideration.append(safety_ethics_1)
    warehouse_scenario_1.hasEthicalConsideration.append(accountability_ethics_1)

# Save ontology
onto.save(file="hi_ontology_extended_populated.owl", format="rdfxml")

In [51]:
from rdflib import Graph

In [65]:
# initialise an empty graph
g = Graph()

# parse a knowledge graph from a local file
g = Graph().parse("hi_ontology_extended.owl")

In [66]:
print(len(g))

581


In [67]:
asserted = Graph()
inferred = Graph()

asserted = asserted.parse("hi_ontology_extended.owl")
DeductiveClosure(OWLRL_Semantics).expand(g) 
inferred = g - asserted

In [68]:
print(len(g))

1263


In [69]:
print(len(inferred))

706
